# 1 Алгоритмы токенизации. BPE, WordPiece, Unigram 

## Task 1-1 BPE

In [ ]:
# Простейшая реализация ядра BPE:
from collections import Counter

In [ ]:
def get_pairs(symbols):
    # Ваш код здесь
    # Возвращает список кортежей соседних пар в списке символов.
    # Пример: ["h","e","l","l","o"] -> 
    # [("h","e"),("e","l"),("l","l"),("l","o")].
    pairs = []
    for i in range(len(symbols)-1):
        pairs.append((symbols[i], symbols[i+1]))
    return pairs

In [ ]:
def merge_pair(symbols, pair):
    # Ваш код здесь
    # Объединяет все вхождения заданной пары в новый токен.
    # Пример: ["h","e","l","l","o"], pair=("l","l") 
    # -> ["h","e","ll","o"].
    new_symbols = []
    i = 0
    while i < len(symbols):
        # Если найдено вхождение пары, объединяем её
        if i < len(symbols)-1 and (symbols[i], symbols[i+1]) == pair:
            new_symbols.append(symbols[i] + symbols[i+1])
            i += 2
        else:
            new_symbols.append(symbols[i])
            i += 1
    return new_symbols

In [ ]:
word = ["a", "b", "a", "b", "c"]
pair = ("a", "b")
print(get_pairs(word))
print(merge_pair(word, pair)) 

## Task 1-2 WordPiece

In [ ]:
def get_pairs_wp(symbols):
    pairs = []
    for i in range(len(symbols)-1):
        pairs.append((symbols[i], symbols[i+1]))
    return pairs

def merge_pair_wp(symbols, pair):
    # Нужно будет дописать в задании самостоятельно 

    new_symbols = []
    i = 0
    while i < len(symbols):
        if i < len(symbols)-1 and (symbols[i], symbols[i+1]) == pair:
            merged = symbols[i] + symbols[i+1]
            if len(new_symbols) > 0:  # если не начало слова
                merged = "##" + merged
            new_symbols.append(merged)
            i += 2
        else:
            new_symbols.append(symbols[i])
            i += 1
    return new_symbols

In [ ]:
# Пример использования:
symbols = ["p", "l", "a", "y"]
print(get_pairs_wp(symbols))  # ожидаем [("p","l"),("l","a"),("a","y")]
print(merge_pair_wp(symbols, ("l","a")))  # ожидаем ["p","##la","y"] 

# Lesson 3 RoBERTa DeBERTa

# Task 3-1

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import torch

/home/russele7/practicum/dle/sprint_5/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Загружаем токенизатор и модель XLM-R для классификации тональности
model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment" # Ваш код здесь
tokenizer = AutoTokenizer.from_pretrained(model_name)# Ваш код здесь  
model = AutoModelForSequenceClassification.from_pretrained(model_name) # Ваш код здесь

In [3]:
# Создаём pipeline для классификации
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer) # Ваш код здесь

Device set to use cuda:0


In [4]:
# Тестовые тексты на разных языках
texts = [
    "This movie is absolutely fantastic! I loved every moment of it.",  # английский
    "Этот фильм просто ужасен, потратил время зря.",  # русский  
    "¡Me encanta este producto! Es increíble y muy útil.",  # испанский
    "I'm not sure about this book, it's okay I guess.",  # английский
    "Сервис отличный, всем рекомендую!",  # русский
    "No me gusta nada, muy decepcionante."  # испанский
]

In [6]:
# Классифицируем тональность для каждого текста
for i, text in enumerate(texts):
    result = classifier(text)# Ваш код здесь
    print(f"Текст {i+1}: {text}")
    print(f"Тональность: {result[0]['label']} (уверенность: {result[0]['score']:.3f})")
    print("-" * 50) 

Текст 1: This movie is absolutely fantastic! I loved every moment of it.
Тональность: positive (уверенность: 0.949)
--------------------------------------------------
Текст 2: Этот фильм просто ужасен, потратил время зря.
Тональность: negative (уверенность: 0.935)
--------------------------------------------------
Текст 3: ¡Me encanta este producto! Es increíble y muy útil.
Тональность: positive (уверенность: 0.950)
--------------------------------------------------
Текст 4: I'm not sure about this book, it's okay I guess.
Тональность: neutral (уверенность: 0.639)
--------------------------------------------------
Текст 5: Сервис отличный, всем рекомендую!
Тональность: positive (уверенность: 0.907)
--------------------------------------------------
Текст 6: No me gusta nada, muy decepcionante.
Тональность: negative (уверенность: 0.952)
--------------------------------------------------


# Task 3-2

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

In [ ]:
# Загружаем модель и токенизатор
model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
tokenizer = AutoTokenizer.from_pretrained(model_name) # Ваш код здесь
model = AutoModelForSequenceClassification.from_pretrained(model_name) # Ваш код здесь
model.eval()

In [9]:
# Сложные пары предложений для анализа с переводами
pairs = [
    {
        "premise_en": "The bank by the river was steep and muddy.",
        "hypothesis_en": "The financial institution was near the water.",
        "premise_ru": "Берег у реки был крутым и грязным.",
        "hypothesis_ru": "Финансовое учреждение было рядом с водой."
    },
    {
        "premise_en": "The doctor advised the lawyer because she felt unwell.",
        "hypothesis_en": "The lawyer was feeling unwell.",
        "premise_ru": "Доктор дал совет адвокату, потому что она плохо себя чувствовала.",
        "hypothesis_ru": "Адвокат плохо себя чувствовал."
    },
    {
        "premise_en": "Despite initial promising results announced in the press conference, the drug failed in clinical trials because of unexpected side effects observed in elderly patients.",
        "hypothesis_en": "Side effects caused the drug to fail.",
        "premise_ru": "Несмотря на первоначальные многообещающие результаты, объявленные на пресс-конференции, препарат провалился в клинических испытаниях из-за неожиданных побочных эффектов, наблюдаемых у пожилых пациентов.",
        "hypothesis_ru": "Побочные эффекты стали причиной провала препарата."
    }
]

In [10]:
# Классифицируем отношения
for pair in pairs:
    # Токенизация и подготовка ввода (используем английские версии!)
    # Используйте tokenizer для преобразования текста в тензоры
    # В токенизаторе передавайте оба предложения и используйте return_tensors="pt", padding=True, truncation=True
    inputs = tokenizer(
        pair['premise_en'], 
        pair['hypothesis_en'],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ) # Ваш код здесь
    
    # Подаём данные в модель
    with torch.no_grad():
        outputs = model(**inputs)# Ваш код здесь
    
    # Преобразуем выходы в вероятности
    probs = F.softmax(outputs.logits, dim=-1) # Ваш код здесь
    probs = probs[0].cpu().numpy()
    
    # Определяем метку с максимальной вероятностью
    labels_ru = ["следствие", "нейтрально", "противоречие"]
    pred_label_ru = labels_ru[probs.argmax()]
    
    # Выводим результат полностью на русском
    print(f"Событие: {pair['premise_ru']}")
    print(f"Гипотеза: {pair['hypothesis_ru']}")
    print(f"Предсказание: {pred_label_ru} (уверенность: {probs.max():.2%})")
    print(f"Вероятности: [следствие: {probs[0]:.2%}, нейтрально: {probs[1]:.2%}, противоречие: {probs[2]:.2%}]")
    print("-" * 80) 

Событие: Берег у реки был крутым и грязным.
Гипотеза: Финансовое учреждение было рядом с водой.
Предсказание: нейтрально (уверенность: 99.85%)
Вероятности: [следствие: 0.10%, нейтрально: 99.85%, противоречие: 0.05%]
--------------------------------------------------------------------------------
Событие: Доктор дал совет адвокату, потому что она плохо себя чувствовала.
Гипотеза: Адвокат плохо себя чувствовал.
Предсказание: следствие (уверенность: 98.67%)
Вероятности: [следствие: 98.67%, нейтрально: 1.23%, противоречие: 0.10%]
--------------------------------------------------------------------------------
Событие: Несмотря на первоначальные многообещающие результаты, объявленные на пресс-конференции, препарат провалился в клинических испытаниях из-за неожиданных побочных эффектов, наблюдаемых у пожилых пациентов.
Гипотеза: Побочные эффекты стали причиной провала препарата.
Предсказание: следствие (уверенность: 98.83%)
Вероятности: [следствие: 98.83%, нейтрально: 1.04%, противоречие: 0.

# 4 Лёгкие модели: rubert_tiny и аналоги

# Task 4-1

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [17]:
def get_model_size_mb(model):
    # Считаем размер всех параметров модели
    param_size = 0
    for param in model.parameters():
        # Количество элементов * размер каждого элемента
        param_size += param.nelement() * param.element_size() # Ваш код здесь
    
    # Считаем размер буферов модели (например, batch norm statistics)
    buffer_size = 0
    for buffer in model.buffers():
        # Аналогично параметрам
        buffer_size += buffer.nelement() * buffer.element_size()# Ваш код здесь
    
    # Переводим из байт в мегабайты
    # (param_size + buffer_size) делить на 1024 дважды
    total_size_mb = (param_size + buffer_size) / 1024 / 1024 # Ваш код здесь
    return total_size_mb

In [18]:
# Тестируем функцию на разных моделях
models_to_test = [
    "distilbert-base-uncased",
    "cointegrated/rubert-tiny",
    "microsoft/MiniLM-L12-H384-uncased"
]

print("Сравнение размеров моделей")
print("=" * 40)

for model_name in models_to_test:
    print(f"\nЗагружаем {model_name}...")
    
    # Загружаем модель
    model = AutoModelForSequenceClassification.from_pretrained(model_name) # Ваш код здесь
    
    # Вычисляем размер
    size_mb = get_model_size_mb(model)
    
    # Считаем количество параметров
    # Сумма всех параметров модели в миллионах
    num_params = sum(p.numel() for p in model.parameters()) / 1e6 # Ваш код здесь
    
    print(f"Размер: {size_mb:.1f} МБ")
    print(f"Параметры: {num_params:.1f}M")
    print("-" * 40) 

Сравнение размеров моделей

Загружаем distilbert-base-uncased...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Размер: 255.4 МБ
Параметры: 67.0M
----------------------------------------

Загружаем cointegrated/rubert-tiny...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Размер: 45.0 МБ
Параметры: 11.8M
----------------------------------------

Загружаем microsoft/MiniLM-L12-H384-uncased...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Размер: 127.3 МБ
Параметры: 33.4M
----------------------------------------


# Task 4-2

In [19]:
import time
from transformers import pipeline

In [21]:
time.time()

1766520446.6194415

In [25]:
def measure_inference_time(classifier, texts, num_runs=10):
    times = []
    for _ in range(num_runs):
        start_time = time.time() # Ваш код здесь
        # Прогоняем все тексты через классификатор
        for text in texts:
            _ = classifier(text) # Ваш код здесь
        end_time = time.time()# Ваш код здесь
        # Сохраняем время выполнения
        times.append(end_time - start_time) # Ваш код здесь)
    
    avg_time = sum(times) / len(times) # Ваш код здесь
    # Среднее время на один текст
    avg_time_per_text = avg_time / len(texts) # Ваш код здесь
    return avg_time_per_text * 1000  # в миллисекундах

In [26]:
# Тестовые тексты на английском
test_texts = [
    "This product is amazing!", # Этот продукт потрясающий!
    "Terrible quality, very disappointed.", # Ужасное качество, очень разочарован.
    "The service was okay, nothing special.", # Сервис был нормальным, ничего особенного.
    "Outstanding experience! Highly recommend!", # Выдающийся опыт! Настоятельно рекомендую!
    "Poor customer support, took forever." # Плохая поддержка клиентов, всё заняло вечность.
]

# Список англоязычных моделей для тестирования
models_to_test = [
    "distilbert-base-uncased",
    "microsoft/MiniLM-L12-H384-uncased"
]

In [27]:
for model_name in models_to_test:
    print(f"\nТестирование скорости: {model_name}")
    print("=" * 50)
    
    # Создаём pipeline для модели
    classifier = pipeline("text-classification", model=model_name)
    
    # Измеряем среднее время инференса
    avg_time = measure_inference_time(classifier, test_texts)
    
    print(f"Среднее время обработки одного текста: {avg_time:.3f} мс")


Тестирование скорости: distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Среднее время обработки одного текста: 69.420 мс

Тестирование скорости: microsoft/MiniLM-L12-H384-uncased


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Среднее время обработки одного текста: 7.638 мс
